# 02 · Selezione e baseline / Selection and baselines

**Input · IT:** Le 15 feature candidate del notebook 01 e `settings.json`.

**Output · IT:** 5–8 input ordinati, preprocessing training-only, K-Means/GMM e `results/baseline_comparison.csv`.

**Input · EN:** The 15 candidate features from Notebook 01 and `settings.json`.

**Output · EN:** 5–8 ordered inputs, training-only preprocessing, K-Means/GMM and `results/baseline_comparison.csv`.

**Prima di iniziare / Before starting:** [scaling e leakage / scaling and leakage](https://scikit-learn.org/stable/common_pitfalls.html), [K-Means](https://scikit-learn.org/stable/modules/clustering.html#k-means), [Gaussian mixtures](https://scikit-learn.org/stable/modules/mixture.html); unità diverse e ridondanza / different units and redundancy.

**Avvio / Start:** Jupyter dalla cartella del progetto; import di `core` dalla radice con `settings.json` / Jupyter from the project folder; import `core` from the root containing `settings.json`.


## A · Selezione e preprocessing / Selection and preprocessing

### 1. Dividere le date / Split the dates

**IT:** Dividete le feature dopo il warmup in training, validation e test con proporzioni 60/20/20. Restituite confini e conteggi ordinati, senza sovrapposizioni né shuffle.

**EN:** Split features after warmup into training, validation and test using 60/20/20 proportions. Return ordered boundaries and counts, with no overlaps or shuffling.


In [1]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# project_root per far girare il notebook su qualunque computer
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

settings = json.loads((project_root / "settings.json").read_text())

# Le 15 feature prodotte dal notebook 01
features = pd.read_csv(project_root / settings["paths"]["results"] / "features.csv",
                       index_col="date", parse_dates=True)
print("Feature caricate:", features.shape, "| dal", features.index[0].date(), "al", features.index[-1].date())

# Divisione cronologica 60/20/20 (dopo il warm-up, già tolto nel notebook 01)
from core.selection import dividi_date

confini = dividi_date(features, settings)
tab_split = pd.DataFrame(confini).T
tab_split["quota"] = (tab_split["n_obs"] / len(features)).round(3)
display(tab_split)

# Etichetta del blocco per ogni giorno, per i controlli
split = pd.Series(index=features.index, dtype="object")
for nome, b in confini.items():
    split.loc[b["start"]:b["end"]] = nome

print("Ogni giorno in un solo blocco:", split.notna().all(),
      "| totale:", tab_split["n_obs"].sum(), "=", len(features))
print("Blocchi in ordine, senza sovrapposizioni:",
      confini["train"]["end"] < confini["validation"]["start"]
      and confini["validation"]["end"] < confini["test"]["start"])
print("Ordine cronologico rispettato (niente shuffle):",
      (split.map({"train": 0, "validation": 1, "test": 2}).diff().dropna() >= 0).all())

Feature caricate: (3617, 15) | dal 2005-11-16 al 2020-04-01


,start,end,n_obs,quota
train,2005-11-16,2014-07-02,2170,0.599945
validation,2014-07-03,2017-05-16,723,0.199889
test,2017-05-17,2020-04-01,724,0.200166


Ogni giorno in un solo blocco: True | totale: 3617 = 3617
Blocchi in ordine, senza sovrapposizioni: True
Ordine cronologico rispettato (niente shuffle): True


### 2. Controllare la variabilità / Check variability

**IT:** Individuate feature costanti o inutilizzabili usando solo il training e criteri coerenti con le unità. Salvate le esclusioni e verificate che modificare validation o test non cambi il risultato.

**EN:** Identify constant or unusable features using training only and unit-aware criteria. Record exclusions and check that changing validation or test leaves the result unchanged.


### 3. Selezionare 5–8 input / Select 5–8 inputs

**IT:** Implementate `selection.select_features` restituendo 5–8 nomi ordinati, sulla base delle correlazioni e della ridondanza training. Conservate al massimo 2–3 sottoinsiemi candidati nei settings; non confrontate likelihood fra input diversi.

**EN:** Implement `selection.select_features` to return 5–8 ordered names based on training correlations and redundancy. Keep at most 2–3 candidate subsets in settings; do not compare likelihoods across different inputs.


### 4. Standardizzare gli input / Scale the inputs

**IT:** Implementate `selection.prepare_model_data` adattando lo scaler soltanto al training. Restituite `feature_names`, `scaler`, `boundaries` e `scaled_blocks` datati (`train`, `validation`, `test`), controllando ordine e valori finiti.

**EN:** Implement `selection.prepare_model_data` by fitting the scaler on training only. Return `feature_names`, `scaler`, `boundaries` and dated `scaled_blocks` (`train`, `validation`, `test`), checking order and finite values.


### 5. Integrare il preprocessing / Integrate preprocessing

**IT:** Portate selezione e scaling in `core/selection.py`; lasciate la lista iniziale vuota finché la selezione non è conclusa. Verificate training-only, cronologia e trasformazioni identiche dopo salvataggio e ricaricamento dello scaler.

**EN:** Move selection and scaling into `core/selection.py`; keep the initial list empty until selection is complete. Check training-only fitting, chronology and identical transformations after saving and reloading the scaler.


## B · Baseline di clustering / Clustering baselines

### 6. Adattare K-Means / Fit K-Means

**IT:** Adattate K-Means agli input training scalati e applicate il modello fisso ai blocchi successivi. Restituite estimatore e label datate, controllando allineamento e ripetibilità con seed fisso.

**EN:** Fit K-Means on scaled training inputs and apply the frozen model to later blocks. Return the estimator and dated labels, checking alignment and repeatability with a fixed seed.


### 7. Adattare la GMM / Fit the GMM

**IT:** Adattate `GaussianMixture` diagonale agli stessi input e restituite estimatore, label e probabilità. Controllate convergenza, varianze positive e somme unitarie, registrando anche i fit falliti.

**EN:** Fit diagonal `GaussianMixture` on the same inputs and return the estimator, labels and probabilities. Check convergence, positive variances and unit row sums, recording failed fits too.


### 8. Confrontare le baseline / Compare the baselines

**IT:** Confrontate i numeri di cluster configurati su training e validation, mantenendo input e date identici. Salvate occupazione, cambi di label e diagnostiche in `results/baseline_comparison.csv`, scegliendo i parametri senza il test.

**EN:** Compare configured cluster counts on training and validation, keeping inputs and dates identical. Save occupancy, label changes and diagnostics to `results/baseline_comparison.csv`, selecting parameters without test data.


### 9. Visualizzare i gruppi / Plot the groups

**IT:** Unite label datate e dati non scalati per produrre profili e timeline. Controllate conteggi, unità e join; i numeri dei cluster non identificano gruppi equivalenti fra modelli.

**EN:** Join dated labels to unscaled data to produce profiles and timelines. Check counts, units and joins; cluster numbers do not identify equivalent groups across models.


### 10. Integrare le baseline / Integrate the baselines

**IT:** Implementate `models.fit_baselines` in `core/models.py` e riutilizzate i grafici di `core/plots.py`. Verificate che i modelli ricaricati riproducano label e probabilità senza nuovo fitting.

**EN:** Implement `models.fit_baselines` in `core/models.py` and reuse charts from `core/plots.py`. Check that reloaded models reproduce labels and probabilities without refitting.
